# MODULE 45 — FACE DETECTION, LANDMARKS & ALIGNMENT

# 45.1 FACE DETECTION

**Current Topic:** Face Detection  
**Priority:** Must Know

---

## 1. What is Face Detection?

Face detection is a Computer Vision task that identifies where human faces are located in an image or video frame.

The important distinction:

- **Face detection** answers: Where is the face?
- **Face recognition** answers: Whose face is it?

For example, suppose you give a system an image containing three people.

![Face detection using MediaPipe showing detected faces with bounding boxes and facial keypoints](https://ai.google.dev/static/edge/mediapipe/images/solutions/examples/face_detector.png)

Source: [Google AI Edge — MediaPipe Face Detector](https://ai.google.dev/edge/mediapipe/solutions/vision/face_detector)

The detector should:

1. Locate all three faces.
2. Draw a bounding box around each face.
3. Assign a confidence score to each detection.
4. Return the detected face coordinates.

**Important:** Face detection does not identify the person's identity. It only detects face regions.

### Face Detection vs Face Recognition

| Face Detection | Face Recognition |
|---|---|
| Locates faces in an image. | Identifies or verifies a person's identity. |
| Produces bounding boxes and detection scores. | Produces identity predictions or similarity scores. |
| Answers WHERE? | Answers WHO? |
| Used as an initial stage in many facial analysis pipelines. | Used in identity verification and identification systems. |

---

## 2. Face Bounding Boxes

A bounding box is a rectangular region surrounding a detected face.

It tells us the position and dimensions of the face in the original image.

![Face bounding boxes illustrating detected face regions](https://res.cloudinary.com/upwork-cloud/image/upload/c_scale%2Cw_1000/v1688967290/catalog/1677744763008335872/yjlzeaziksik3iiqnpqk.jpg)

Source: [Upwork — Object/Image Detection using OpenCV](https://www.upwork.com/services/product/development-it-object-image-detection-using-opencv-1677744763008335872)

A common representation is:

$$
B = (x_{\min}, y_{\min}, x_{\max}, y_{\max})
$$

Where:

| Parameter | Meaning |
|---|---|
| \\(x_{\min}\\) | Left coordinate |
| \\(y_{\min}\\) | Top coordinate |
| \\(x_{\max}\\) | Right coordinate |
| \\(y_{\max}\\) | Bottom coordinate |

Another representation uses:

$$
B = (x,y,w,h)
$$

- \\(x,y\\): Top-left corner.
- \\(w\\): Bounding-box width.
- \\(h\\): Bounding-box height.

### Example

Suppose a face detector returns:

```python
bbox = (100, 80, 250, 260)
```

This means:

- Left = 100
- Top = 80
- Right = 250
- Bottom = 260

Therefore:

$$
Width = 250 - 100 = 150
$$

$$
Height = 260 - 80 = 180
$$

The face occupies a rectangular region of 150 × 180 pixels.

### Important: Understanding Bounding Box Coordinates

For an image:

- The origin \\((0,0)\\) is at the top-left corner.
- The X-coordinate increases towards the right.
- The Y-coordinate increases towards the bottom.

For example:

```text
(0,0) --------------------> X
  |
  |
  |
  |
  |
  v
  Y
```

The bounding box specifies the rectangular region that contains the detected face.

---

## 3. Detection Confidence

A face detector does not simply say whether a face exists. It also produces a confidence score indicating how strongly the model believes that a detected region contains a face.

Example:

```python
detections = [
    {"bbox": (100, 80, 250, 260), "confidence": 0.98},
    {"bbox": (300, 90, 420, 230), "confidence": 0.87},
    {"bbox": (500, 100, 600, 200), "confidence": 0.42}
]
```

### Interpretation

| Detection | Confidence | Interpretation |
|---|---:|---|
| Face 1 | 0.98 | Strong detection |
| Face 2 | 0.87 | Strong detection |
| Face 3 | 0.42 | Less certain detection |

We can introduce a confidence threshold:

$$
Threshold = 0.70
$$

Only detections with confidence greater than or equal to 0.70 are retained.

```python
threshold = 0.70

filtered = [
    detection
    for detection in detections
    if detection["confidence"] >= threshold
]
```

### Why is this important?

A detector might identify a face-like region in a photograph that is actually a poster, drawing or background object.

Confidence thresholding helps control these detections.

**Technical note:** Confidence scores are model-specific scores, not necessarily calibrated probabilities of correctness.

A confidence score of 0.98 does not automatically mean that the detection has exactly a 98% probability of being correct.

### Understanding the Confidence Threshold

| Threshold | Effect |
|---|---|
| Low threshold | Retains more detections but may allow more false positives. |
| High threshold | Removes more uncertain detections but may reject some genuine faces. |

There is a trade-off between detecting as many genuine faces as possible and avoiding incorrect detections.

---

## 4. Multiple Face Detection

A face detection system must handle images containing multiple people.

For example:

![Multiple face detection with individual bounding boxes around several people](https://user-images.githubusercontent.com/22003608/82897863-b27bef00-9f2e-11ea-983e-cec5fff8bc15.jpg)

Source: [OpenCV Real-Time Face Detection — GitHub](https://github.com/cpatrickalves/cv-realtime-face-detection)

The model should return multiple bounding boxes rather than just one.

Mathematically, its output can be represented as:

$$
D = \{B_1,B_2,\ldots,B_n\}
$$

Where:

- \\(D\\) = Set of detections.
- \\(B_i\\) = Bounding box of an individual face.
- \\(n\\) = Number of detected faces.

### Real-world challenges

- Faces partially hidden behind other people.
- Faces at different distances.
- Side-facing faces.
- Small faces in crowded images.
- Different lighting conditions.

For example, a face detection system used in a video conferencing application must detect every visible participant, not just the person closest to the camera.

### Practical Example

Suppose an image contains five people.

The detector might return:

```python
detections = [
    {"bbox": (50, 80, 150, 190), "confidence": 0.98},
    {"bbox": (180, 70, 280, 180), "confidence": 0.95},
    {"bbox": (320, 90, 410, 200), "confidence": 0.91},
    {"bbox": (450, 60, 530, 150), "confidence": 0.89},
    {"bbox": (570, 100, 640, 180), "confidence": 0.85}
]
```

The system has detected five individual faces.

Each detection contains:

1. Bounding box coordinates.
2. Confidence score.

---

## 5. False Positives

A false positive occurs when the detector identifies something as a face even though it is not a real face.

![Examples of face-like patterns and visual structures](https://adam.harvey.studio/what-is-a-face/)

Source: [Adam Harvey — What Is a Face?](https://adam.harvey.studio/what-is-a-face/)

Examples:

- A face-like pattern on a wall.
- A portrait in a photograph.
- A cartoon character.
- An object that resembles facial features.

### Confusion Matrix Connection

Let's connect this with the confusion matrix you already know from Deep Learning.

| Actual condition | Predicted face | Predicted non-face |
|---|---|---|
| Face | True Positive (TP) | False Negative (FN) |
| Non-face | False Positive (FP) | True Negative (TN) |

### Two important metrics

#### 1. Precision

$$
Precision = \frac{TP}{TP+FP}
$$

Measures how many detected faces are actually faces.

#### 2. Recall

$$
Recall = \frac{TP}{TP+FN}
$$

Measures how many actual faces the detector successfully finds.

### Understanding Precision and Recall

| Metric | Focus | What does it measure? |
|---|---|---|
| Precision | Correctness of positive detections | How many predicted faces are genuine faces? |
| Recall | Coverage of actual faces | How many genuine faces were detected? |

For security applications, both are important. Missing a face and incorrectly detecting a face can create different problems.

---

## 6. Practical Implementation Perspective

In an actual face detection pipeline, the process looks like this:

![Face detection and facial analysis pipeline reference](https://miro.medium.com/v2/resize%3Afit%3A1400/1%2AsDkWhhylAZuHQpfIAxIikg.png)

Source: [Face Detection in 5 Minutes — Medium](https://medium.com/augmented-startups/face-detection-in-5-minutes-70-fps-on-cpu-690efd10bbae)

### Face Detection Pipeline

```text
Input Image
     ↓
Preprocessing
     ↓
Pretrained Face Detector
     ↓
Bounding Box Predictions
     ↓
Confidence Filtering
     ↓
Detected Face Regions
```

We will implement actual face detection in the practical stage using a pretrained detector rather than training a face detector from scratch.

### Understanding Each Stage

| Stage | Description |
|---|---|
| Input Image | Image supplied to the detection system. |
| Preprocessing | Image preparation before model inference. |
| Pretrained Face Detector | Model predicts possible face locations. |
| Bounding Box Predictions | Returns the coordinates of detected faces. |
| Confidence Filtering | Removes detections below the selected confidence threshold. |
| Detected Face Regions | Final face regions retained after filtering. |

### Visualizing a Predicted Bounding Box Using OpenCV

For now, understand how to visualize a predicted bounding box:

```python
import cv2

image = cv2.imread("face.jpg")

# Example detector output
x1, y1, x2, y2 = 100, 80, 250, 260
confidence = 0.98

cv2.rectangle(
    image,
    (x1, y1),
    (x2, y2),
    (0, 255, 0),
    2
)

cv2.putText(
    image,
    f"Face: {confidence:.2f}",
    (x1, y1 - 10),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.6,
    (0, 255, 0),
    2
)

cv2.imshow("Face Detection", image)

cv2.waitKey(0)
cv2.destroyAllWindows()
```

### Understanding the OpenCV Functions

| Function | Purpose |
|---|---|
| `cv2.imread()` | Loads an image from the specified path. |
| `cv2.rectangle()` | Draws a rectangular bounding box around the detected face. |
| `cv2.putText()` | Displays the confidence score on the image. |
| `cv2.imshow()` | Displays the resulting image. |
| `cv2.waitKey(0)` | Waits for a keyboard input. |
| `cv2.destroyAllWindows()` | Closes the OpenCV windows. |

**Important:** This code visualizes a detection; it assumes the bounding box and confidence have already been returned by a detector.

---

## What You Should Remember 🧠

1. Face detection identifies the location of faces, not their identities.
2. Bounding boxes represent face locations and dimensions.
3. Confidence scores indicate the detector's certainty.
4. Confidence thresholds filter uncertain detections.
5. Multiple face detection returns multiple bounding boxes.
6. False positives are non-face regions incorrectly detected as faces.
7. Precision and recall help evaluate detection performance.

---

## Technical Clarifications & Additional Engineering Notes

These clarifications connect the concepts above with actual Computer Vision implementations.

### 1. Pixel Coordinates vs Normalized Coordinates

Bounding boxes can be represented using pixel coordinates or normalized coordinates.

**Pixel coordinates:**

```python
bbox = (100, 80, 250, 260)
```

These coordinates directly refer to pixel locations in the image.

**Normalized coordinates:**

Coordinates are represented relative to image dimensions, generally between 0 and 1.

For an image with width \\(W\\) and height \\(H\\):

$$
x_{normalized} = \frac{x_{pixel}}{W}
$$

$$
y_{normalized} = \frac{y_{pixel}}{H}
$$

To convert back:

$$
x_{pixel} = x_{normalized} \times W
$$

$$
y_{pixel} = y_{normalized} \times H
$$

This distinction is important because different face detection frameworks may return different coordinate formats.

For example, MediaPipe provides normalized bounding box coordinates in some APIs, while its newer Python Face Detector task reports bounding box origin and dimensions in pixels.

Reference: [MediaPipe Face Detector — Python](https://developers.google.com/edge/mediapipe/solutions/vision/face_detector/python)

### 2. Bounding Box Format Matters

Consider:

```python
bbox_xyxy = (100, 80, 250, 260)
```

This represents:

```text
(x_min, y_min, x_max, y_max)
```

However, another model might return:

```python
bbox_xywh = (100, 80, 150, 180)
```

This represents:

```text
(x, y, width, height)
```

Never assume that two detection models return the same bounding box format.

Always inspect the model documentation before processing its output.

### 3. Face Detection Can Also Return Keypoints

Some modern face detectors return more than bounding boxes.

For example, MediaPipe Face Detector can return six facial keypoints for each detected face, including:

- Left eye.
- Right eye.
- Nose tip.
- Mouth center.
- Left ear tragion.
- Right ear tragion.

These keypoints provide additional geometric information about the face.

This creates the connection between face detection and the next topic: **Facial Landmarks**.

Reference: [Google AI Edge — Face Detector](https://ai.google.dev/edge/mediapipe/solutions/vision/face_detector)

---